# PlanForge
> Your AI business advisor. Tell it what you want to achieve, have a conversation, get a plan.

*v0.5.0*

In [ ]:
# @title Setup
!pip install -q pysolvr-client
import sys, json as _json, time as _time
from google.colab import userdata, drive
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    display(HTML('<div style="color:#f87171;padding:12px;border:1px solid #f87171;border-radius:8px;font-family:Inter,sans-serif"><b>API key not found</b><ol><li>Click key icon in left sidebar</li><li>Add: PLANFORGE_API_KEY</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'PLANFORGE_API_KEY not found'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'exports', 'uploads'])
drive_mgr.ensure_folders()

# Session state
_state = {'goal': None, 'goal_info': {}, 'topics': [], 'current_topic': None, 'history': [], 'locked': {}}

if client.health_check():
    ui.success('Connected', f'Drive: {drive_mgr.root}')
else:
    ui.error('Connection failed', 'Check API key')


In [ ]:
# @title What do you want to achieve?
GOAL = 'I need funding for my startup'  # @param ["I need funding for my startup", "I need a UK visa for my business", "I want to validate my idea", "I need a bank loan", "I'm applying to an accelerator", "I need a grant (Innovate UK)", "I just want a quick viability check"]

_goal_map = {
    'I need funding for my startup': 'seed_pitch',
    'I need a UK visa for my business': 'innovator_visa',
    'I want to validate my idea': 'validate',
    'I need a bank loan': 'bank_loan',
    "I'm applying to an accelerator": 'accelerator',
    'I need a grant (Innovate UK)': 'innovate_uk',
    'I just want a quick viability check': 'scorecard',
}
_state['goal'] = _goal_map.get(GOAL, 'scorecard')
_state['history'] = []
_state['locked'] = {}

# Fetch topics for this goal
result = client.call('GET', '/goals')
if result['ok']:
    goals = {g['id']: g for g in result['data']['goals']}
    goal_info = goals.get(_state['goal'], {})
    _state['goal_info'] = goal_info
    _state['topics'] = [{'id': t['id'], 'name': t['name']} for t in goal_info.get('required_topics', [])]
    _state['current_topic'] = _state['topics'][0]['id'] if _state['topics'] else None
    names = [t['name'] for t in _state['topics']]
    ui.card(f'Goal: {goal_info.get("name", GOAL)}',
            f'<b>Audience:</b> {goal_info.get("audience", "you")}<br>'
            f'<b>Topics to cover:</b> {", ".join(names)}<br><br>'
            f'Run the next cell to start your discovery conversation.',
            status='success')
else:
    ui.success(f'Goal set: {GOAL}', 'Run the next cell to start chatting')


In [ ]:
# @title Discovery Chat
# Run this cell once - the chat interface appears below.

# --- Widget layout ---
_progress_out = widgets.Output()
_chat_out = widgets.Output(layout={'border': '1px solid #334155', 'border_radius': '12px',
                                    'padding': '16px', 'max_height': '400px',
                                    'overflow_y': 'auto'})
_input_area = widgets.Textarea(placeholder='Type your message or paste text (CV, financials, notes)...',
                               layout=widgets.Layout(width='100%', height='80px'))
_send_btn = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='80px', height='36px'))
_lock_btn = widgets.Button(description='Lock this in', button_style='success',
                            layout=widgets.Layout(width='120px', height='36px', display='none'))
_status_out = widgets.Output()


def _render_progress():
    with _progress_out:
        clear_output(wait=True)
        if not _state['topics']:
            return
        locked = _state['locked']
        current = _state['current_topic']
        done = sum(1 for t in _state['topics'] if t['id'] in locked)
        total = len(_state['topics'])
        pct = int(done / total * 100) if total else 0
        # Progress bar
        bar = f'<div style="background:#1e293b;border-radius:8px;padding:12px;font-family:Inter,sans-serif;margin-bottom:8px">'
        bar += f'<div style="font-size:12px;color:#94a3b8;margin-bottom:6px">{done}/{total} topics covered</div>'
        bar += f'<div style="background:#334155;border-radius:4px;height:6px;margin-bottom:10px"><div style="background:#10b981;height:6px;border-radius:4px;width:{pct}%"></div></div>'
        # Topic pills
        for t in _state['topics']:
            tid, name = t['id'], t['name']
            if tid in locked:
                style = 'background:#065f46;color:#6ee7b7;border:1px solid #10b981'
                icon = '\u2713 '
            elif tid == current:
                style = 'background:#1e40af;color:#93c5fd;border:1px solid #3b82f6'
                icon = '\u25cf '
            else:
                style = 'background:#1e293b;color:#64748b;border:1px solid #334155'
                icon = ''
            bar += f'<span data-topic="{tid}" style="{style};padding:4px 10px;border-radius:12px;font-size:11px;margin:2px 4px 2px 0;display:inline-block;cursor:pointer">{icon}{name}</span>'
        # Ready message
        if done == total:
            bar += f'<div style="color:#10b981;font-size:13px;margin-top:8px;font-weight:600">\u2705 Ready to generate your plan!</div>'
        bar += '</div>'
        display(HTML(bar))


def _render_chat():
    with _chat_out:
        clear_output(wait=True)
        if not _state['history']:
            # Show opening question for current topic
            display(HTML('<p style="color:#94a3b8;font-family:Inter,sans-serif;font-size:13px">'
                         'Tell me about your business. Type a message or paste a document.</p>'))
            return
        html = ''
        for msg in _state['history']:
            if msg.get('type') == 'divider':
                html += f'<div style="text-align:center;margin:12px 0;font-size:11px;color:#64748b">\u2500\u2500 {msg["content"]} \u2500\u2500</div>'
            elif msg['role'] == 'user':
                text = msg['content'][:500] + ('...' if len(msg['content']) > 500 else '')
                html += (f'<div style="text-align:right;margin:6px 0">'
                         f'<span style="background:#1e40af;color:#f1f5f9;padding:8px 12px;'
                         f'border-radius:14px 14px 4px 14px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{text}</span></div>')
            else:
                html += (f'<div style="text-align:left;margin:6px 0">'
                         f'<span style="background:#1e293b;color:#e2e8f0;padding:8px 12px;'
                         f'border-radius:14px 14px 14px 4px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{msg["content"]}</span></div>')
        display(HTML(html))


def _switch_topic(topic_id):
    if topic_id == _state['current_topic']:
        return
    old_name = next((t['name'] for t in _state['topics'] if t['id'] == _state['current_topic']), '')
    new_name = next((t['name'] for t in _state['topics'] if t['id'] == topic_id), '')
    _state['current_topic'] = topic_id
    _state['history'].append({'type': 'divider', 'role': 'system', 'content': f'Switched to: {new_name}'})
    _lock_btn.layout.display = 'none'
    _render_progress()
    _render_chat()


# Topic pill buttons (since HTML onclick won't work in Colab, use real buttons)
_topic_buttons = []
def _make_topic_buttons():
    global _topic_buttons
    _topic_buttons = []
    for t in _state['topics']:
        tid = t['id']
        btn = widgets.Button(description=t['name'], layout=widgets.Layout(height='28px'),
                             button_style='info' if tid == _state['current_topic'] else '')
        btn._topic_id = tid
        btn.on_click(lambda b: _switch_topic(b._topic_id))
        _topic_buttons.append(btn)


def _on_send(btn):
    message = _input_area.value.strip()
    if not message:
        with _status_out:
            clear_output(wait=True)
            display(HTML('<span style="color:#f87171;font-size:12px">Type a message</span>'))
        return
    _state['history'].append({'role': 'user', 'content': message})
    _input_area.value = ''
    _lock_btn.layout.display = 'none'
    _render_chat()
    with _status_out:
        clear_output(wait=True)
        display(HTML('<span style="color:#94a3b8;font-size:12px">Thinking...</span>'))
    # Build request
    topic = _state['current_topic'] or 'business_overview'
    body = {'topic': topic, 'message': message, 'goal_type': _state['goal'],
            'conversation_history': [{'role': m['role'], 'content': m['content'][:1000]}
                                     for m in _state['history'][-6:] if m.get('type') != 'divider'],
            'locked_topics': _state['locked']}
    # Submit + poll
    submit = client.call('POST', '/chat', body)
    if not submit['ok']:
        _state['history'].append({'role': 'assistant', 'content': f"Error: {submit.get('error', 'Failed')}"})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    job_id = submit['data'].get('job_id')
    if not job_id:
        reply = submit['data'].get('response', '')
        _state['history'].append({'role': 'assistant', 'content': reply})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    # Poll
    for _ in range(60):
        _time.sleep(2)
        poll = client.call('GET', f'/jobs/{job_id}')
        if not poll['ok']:
            break
        status = poll['data'].get('status')
        if status == 'complete':
            result_data = poll['data'].get('result', {})
            reply = result_data.get('response', '')
            _state['history'].append({'role': 'assistant', 'content': reply})
            # Check if LLM suggests locking
            topic_status = result_data.get('topic_status', '')
            if topic_status in ('suggested_lock', 'locked'):
                _lock_btn.layout.display = 'inline-block'
                if topic_status == 'locked':
                    _do_lock()
            with _status_out:
                clear_output(wait=True)
            _render_chat()
            return
        elif status == 'failed':
            _state['history'].append({'role': 'assistant', 'content': f"Error: {poll['data'].get('error', 'Failed')}"})
            break
    with _status_out:
        clear_output(wait=True)
    _render_chat()


def _do_lock(btn=None):
    topic = _state['current_topic']
    if not topic:
        return
    # Send lock_in message
    body = {'topic': topic, 'message': 'lock_in', 'goal_type': _state['goal'],
            'conversation_history': [], 'locked_topics': _state['locked']}
    submit = client.call('POST', '/chat', body)
    # Mark as locked locally
    _state['locked'][topic] = 'locked'
    _lock_btn.layout.display = 'none'
    # Auto-advance to next unlocked topic
    for t in _state['topics']:
        if t['id'] not in _state['locked']:
            _switch_topic(t['id'])
            return
    # All done
    _render_progress()
    _render_chat()


_send_btn.on_click(_on_send)
_lock_btn.on_click(_do_lock)
_make_topic_buttons()
_render_progress()
_render_chat()

display(widgets.VBox([
    _progress_out,
    widgets.HBox(_topic_buttons, layout=widgets.Layout(flex_flow='row wrap', margin='0 0 8px 0')),
    _chat_out,
    widgets.HBox([_input_area, widgets.VBox([_send_btn, _lock_btn])]),
    _status_out,
]))


In [ ]:
# @title Generate my plan
result = client.run_async('/generate', {'goal_type': _state['goal']})
if result['ok']:
    data = result['data']
    plan_id = data.get('plan_id', '')
    content = data.get('content', data.get('plan', ''))
    ui.card('Your Plan', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{content}</pre>', status='success')
    if plan_id:
        filename = f'{_state["goal"]}_{plan_id[:8]}.md'
        drive_mgr.save('plans', filename, content, meta={'plan_id': plan_id, 'goal': _state['goal']})
        ui.success('Saved to Drive', f'plans/{filename}')
else:
    ui.error(result.get('error', 'Generation failed'), f'Status: {result.get("status")}')


In [ ]:
# @title Export to a different format
FORMAT = 'executive_summary'  # @param ["executive_summary", "pitch_deck", "one_pager", "financial_summary", "full_pdf"]

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    result = client.run_async('/export', {'content': content, 'format': FORMAT})
    if result['ok']:
        data = result['data']
        ui.card(f'Export: {FORMAT}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
        filename = f'{FORMAT}_{plan_files[-1].replace(".md", "")}.md'
        drive_mgr.save('exports', filename, data['content'], meta={'format': FORMAT})
        ui.success('Saved', f'exports/{filename}')
    else:
        ui.error(result.get('error', 'Export failed'))


In [ ]:
# @title Refine a section
SECTION = ''  # @param {type:"string"}
FEEDBACK = ''  # @param {type:"string"}

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
elif not SECTION.strip():
    ui.error('Enter a section name', 'e.g. Market Analysis, Financial Projections')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    body = {'content': content, 'section': SECTION}
    if FEEDBACK.strip():
        body['feedback'] = FEEDBACK
    result = client.run_async('/refine', body)
    if result['ok']:
        data = result['data']
        ui.card(f'Refined: {data["section"]}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
    else:
        ui.error(result.get('error', 'Refinement failed'))


In [ ]:
# @title My plans
result = client.call('GET', '/plans')
if result['ok']:
    plans = result['data'].get('plans', [])
    if not plans:
        ui.card('Plans', 'No plans generated yet.')
    else:
        rows = [{'ID': p['plan_id'][:8], 'Goal': p.get('goal_type', ''), 'Created': p.get('created_at', '')[:10]} for p in plans]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch plans'))


In [ ]:
# @title Account
result = client.call('GET', '/account')
if result['ok']:
    d = result['data']
    ui.card('Account', (
        f"<b>Plan:</b> {d.get('plan', 'free')}<br>"
        f"<b>Spend:</b> ${d.get('monthly_spend_usd', 0):.2f} / ${d.get('monthly_limit_usd', 0):.2f}<br>"
        f"<b>Top-up:</b> ${d.get('topup_balance_usd', 0):.2f}"
    ), status='success')
else:
    ui.error(result.get('error', 'Could not fetch account'))


In [ ]:
# @title Usage
result = client.get_usage()
if result['ok']:
    data = result['data']
    ui.usage_bar(data.get('current_month_spend_usd', 0), data.get('monthly_limit_usd', 1))
    if data.get('recent'):
        ui.table(data['recent'])
else:
    ui.error(result.get('error', 'Could not fetch usage'), 'Check your API key')
